# batch_ingest-nyc_taxi-spark-iceberg

## 1. Overview

## 2. Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.remote("sc://spark-connect:15002").getOrCreate()

## 3. Read

In [ ]:
raw = spark.read.parquet("s3a://landing/nyc_taxi/")
raw.printSchema()

## 4. Transform

In [ ]:
bronze = (raw
  .where(F.col('tpep_pickup_datetime').isNotNull() & (F.col('passenger_count') > 0))
  .withColumn('trip_date', F.to_date('tpep_pickup_datetime')))

## 5. Write

In [ ]:
bronze.writeTo("lakehouse.bronze.nyc_taxi_trips").using("iceberg").createOrReplace()

## 6. Verify

In [ ]:
spark.table("lakehouse.bronze.nyc_taxi_trips").count()